In [80]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [140]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def ranktoset (A):
    A = list(A)
    sets = [[[A[0]]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append([new])
    return(sets)

def makeset (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    added = []
    for i in range(N):
        new = B[0:i+1]
        N_sets = len(A[i])
        for k in range(N_sets):
            if len(np.intersect1d(A[i][k],new))==len(new):
                break
            if k == N_sets-1:
                A[i].append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(a<= 1)
    constraints.append(a>=0)
    constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1) - (1-cp.sum(a))*r_f + z4 + z2 <= c)
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f <= c)

In [141]:
def cutting_plane(R,r,c,p,m,r_f,sets):
    nonstop = True
    iterations = 1
    while nonstop == True:
        realsets = convertlist(sets)
        [a,obj] = solvenominal(realsets,p,R,r,m,r_f,c)
        newrank = np.argsort(R.dot(a))
        [sets,added] = makeset(sets, newrank)
        if robustcheck(a,R,r,c,p,m,r_f) == True:
            return(a,obj,iterations)
        iterations = iterations + 1
    

In [144]:
N=6
p = np.random.rand(N)
p = p/sum(p)
I = 5
R = np.random.rand(N,I)*3-1
print(R.transpose().dot(p))

[0.15100172 0.56803262 0.30481932 1.24241288 1.40029312]


In [145]:
r = 0.1
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 10
sets = ranktoset(np.arange(N))
print(cutting_plane(R,r,c,p,m,r_f,sets))
convertlist(sets)

(array([7.68319852e-10, 9.38456979e-10, 7.19204807e-10, 3.55058760e-09,
       9.99999998e-01]), 1.400293119620803, 1)


[[0],
 [3],
 [0, 1],
 [3, 5],
 [0, 1, 2],
 [3, 5, 1],
 [0, 1, 2, 3],
 [3, 5, 1, 0],
 [0, 1, 2, 3, 4],
 [3, 5, 1, 0, 2],
 [0, 1, 2, 3, 4, 5]]

In [146]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
solvenominal (psets,p,R,r,m,r_f,c)

(array([4.39483627e-09, 4.45080395e-09, 6.00280914e-09, 5.21410590e-08,
        9.99999963e-01]),
 1.4002931344344907)